<a href="https://colab.research.google.com/github/franciscogarate/upro/blob/main/notebooks/6_Concentracion_500m_California.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculo del punto central con la mayor concentración geográfica

In [33]:
!git clone https://github.com/franciscogarate/upro

fatal: destination path 'upro' already exists and is not an empty directory.


In [6]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

Cargamos los datos:

In [31]:
#fichero = '212827-0-administracion-justicia-csv.csv'
fichero = '201544-3-centros-salud-csv.csv'

In [32]:
df = pd.read_csv(f'upro/data/{fichero}', sep=';', encoding='latin-1', usecols=['NOMBRE','LATITUD','LONGITUD'])
df.dropna(inplace=True)
df['SUMA_ASEG'] = 1000
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'upro/data/201544-3-centros-salud-csv.csv'

Primero, centro el mapa en Madrid

In [26]:
from folium import Map, Marker, plugins
from folium.plugins import HeatMap
map_madrid = Map(location=[40.42, -3.70], zoom_start = 12)

Creamos una matriz de coordenadas (lat, lon):

In [11]:
coords = df[['LATITUD', 'LONGITUD']].to_numpy()

Muestro el mapa de calor

In [12]:
coord_riesgos = np.asarray(coords)
HeatMap(coord_riesgos, radius=20).add_to(map_madrid)
map_madrid

Calculamos la matriz de distancias en grados

In [13]:
distancias = cdist(coords, coords, metric='euclidean')
distancias

array([[0.        , 0.00810279, 0.        , ..., 0.00992206, 0.01396541,
        0.00838166],
       [0.00810279, 0.        , 0.00810279, ..., 0.00197101, 0.0094223 ,
        0.00120975],
       [0.        , 0.00810279, 0.        , ..., 0.00992206, 0.01396541,
        0.00838166],
       ...,
       [0.00992206, 0.00197101, 0.00992206, ..., 0.        , 0.00827622,
        0.00258091],
       [0.01396541, 0.0094223 , 0.01396541, ..., 0.00827622, 0.        ,
        0.01053416],
       [0.00838166, 0.00120975, 0.00838166, ..., 0.00258091, 0.01053416,
        0.        ]])

Sumamos las viviendas de todos los distritos a menos de 500 metros

In [14]:
matrix_cumulos_500m = distancias < 500 / 100000  # Aproximación de 500 metros a 0.005 grados
matrix_cumulos_500m

array([[ True, False,  True, ..., False, False, False],
       [False,  True, False, ...,  True, False,  True],
       [ True, False,  True, ..., False, False, False],
       ...,
       [False,  True, False, ...,  True, False,  True],
       [False, False, False, ..., False,  True, False],
       [False,  True, False, ...,  True, False,  True]])

In [18]:
matrix_capitales = matrix_cumulos_500m * df['SUMA_ASEG'].values[np.newaxis, :]
matrix_capitales

array([[1000,    0, 1000, ...,    0,    0,    0],
       [   0, 1000,    0, ..., 1000,    0, 1000],
       [1000,    0, 1000, ...,    0,    0,    0],
       ...,
       [   0, 1000,    0, ..., 1000,    0, 1000],
       [   0,    0,    0, ...,    0, 1000,    0],
       [   0, 1000,    0, ..., 1000,    0, 1000]])

In [20]:
capital_concentrado = matrix_capitales.sum(axis=1)

Encontramos el índice del punto con mayor concentración

In [21]:
indice_max = np.argmax(capital_concentrado)
punto_central = df.iloc[indice_max]
punto_central

,16
NOMBRE,Fiscalía - Delitos Económicos
LATITUD,40.463543
LONGITUD,-3.691771
SUMA_ASEG,1000


Coordenadas del punto más denso en contador de viviendas

In [30]:
print(f'Latitud: {punto_central['LATITUD']}, Longitud: {punto_central['LONGITUD']}')
print(f'Capital total radio 500 m: {capital_concentrado[indice_max]:,.0f}')
print(f'Número de viviendas del cúmulo: {matrix_cumulos_500m[indice_max].sum()}')

Latitud: 40.46354276590144, Longitud: -3.6917713300891353
Capital total radio 500 m: 14,000
Número de viviendas del cúmulo: 14


In [29]:
Marker(location=[punto_central['LATITUD'], punto_central['LONGITUD']]).add_to(map_madrid)

map_madrid